In [1]:
!pwd

/content


In [2]:
# Need to use darts==0.34.0
!uv pip install darts==0.34.0

!uv pip install -U scikit-learn
!uv pip install pyyaml
!uv pip install catboost
!uv pip install dask[dataframe]

Using Python 3.12.13 environment at: /usr
Resolved 73 packages in 1.03s
Prepared 17 packages in 552ms
Uninstalled 2 packages in 39ms
Installed 17 packages in 36ms
 + adagio==0.2.6
 + coreforecast==0.0.17
 + darts==0.34.0
 + fugue==0.9.7
 + lightning-utilities==0.15.3
 + nfoursid==1.0.2
 - numpy==2.0.2
 + numpy==1.26.4
 + pmdarima==2.1.1
 + pyod==2.1.0
 + pytorch-lightning==2.6.1
 - shap==0.51.0
 + shap==0.49.1
 + statsforecast==2.0.1
 + tbats==1.1.3
 + tensorboardx==2.6.5
 + torchmetrics==1.9.0
 + triad==1.0.2
 + utilsforecast==0.2.15
Using Python 3.12.13 environment at: /usr
Resolved 5 packages in 187ms
Prepared 3 packages in 1.13s
Uninstalled 3 packages in 103ms
Installed 3 packages in 43ms
 - numpy==1.26.4
 + numpy==2.4.4
 - scikit-learn==1.6.1
 + scikit-learn==1.8.0
 - scipy==1.16.3
 + scipy==1.17.1
Using Python 3.12.13 environment at: /usr
Checked 1 package in 77ms
Using Python 3.12.13 environment at: /usr
Resolved 19 packages in 219ms
Prepared 1 package in 1.59s
Installed 1 packa

In [2]:
!pip show darts
!pip show scikit-learn
!pip show pyyaml
!pip show catboost
!pip show dask

Name: darts
Version: 0.34.0
Summary: A python library for easy manipulation and forecasting of time series.
Home-page: https://unit8co.github.io/darts/
Author: 
Author-email: 
License: Apache License 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: holidays, joblib, matplotlib, narwhals, nfoursid, numpy, pandas, pmdarima, pyod, pytorch-lightning, requests, scikit-learn, scipy, shap, statsforecast, statsmodels, tbats, tensorboardX, torch, tqdm, typing-extensions, xarray, xgboost
Required-by: 
Name: scikit-learn
Version: 1.8.0
Summary: A set of python modules for machine learning and data mining
Home-page: https://scikit-learn.org
Author: 
Author-email: 
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: joblib, numpy, scipy, threadpoolctl
Required-by: darts, esda, fastai, hdbscan, imbalanced-learn, libpysal, librosa, mapclassify, mlxtend, pmdarima, pynndescent, pyod, pysal, segregation, sentence-transformers, shap, sklearn-compat, sklearn-pandas, spopt,

**Forecasts**
 - **Country: Canada**
 - **Forecast Horizons:12M and 24M Forecast**

In [3]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import LightGBMModel, CatBoostModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict

class canadaCBForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      """
      Initialize the CatBoost forecast generator for canada data

      Args:
          data_path: Path to the canada data CSV file
          random_seed: Random seed for reproducibility
      """
      # Set random seeds
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      # Load data
      self.data = pd.read_csv(data_path)

      # Define target and exogenous variables
      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      """
      Prepare training data based on forecast horizon

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Tuple of target and exogenous TimeSeries objects
      """
      # Split data based on forecast horizon
      train = self.data.head(-forecast_horizon).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      # Stack target variables
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(train[var]))

      # Stack exogenous variables
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> CatBoostModel:
      """
      Create and return the CatBoost model with horizon-specific parameters

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Configured CatBoostModel
      """
      return CatBoostModel(
          lags=3, # forecast_horizon,
          lags_past_covariates=3, # forecast_horizon,
          output_chunk_length=forecast_horizon,
          random_state=0
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      """
      Generate forecasts for specified horizons

      Args:
          forecast_horizons: List of forecast horizons (e.g., [12, 24])

      Returns:
          Dictionary with forecast horizons as keys and forecast DataFrames as values
      """
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Create and train model
          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              # verbose=True
          )

          # Generate forecast
          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(n=horizon)

          # Convert to DataFrame
          pred_df = pd.DataFrame({
              'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
              'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
              'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
              'forecast_eer': pred.pd_dataframe().iloc[:, 3],
              'forecast_ir': pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          # Save forecast to CSV
          output_file = f'canada_catboost_forecasts_{horizon}m.csv'
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts

def main():
  """
  Main function to run the LGBM forecast generation
  """
  # Initialize forecast generator
  generator = canadaCBForecastGenerator(
      data_path='all_mulvar_data_canada_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"canada_catboost_forecasts_{horizon}m.csv")

if __name__ == "__main__":
  main()

# Created/Modified files during execution:
# canada_catboost_forecasts_12m.csv
# canada_catboost_forecasts_24m.csv


Generating 12-month forecast...
Training model for 12-month horizon...


Generating 12-month predictions...
Forecast saved to canada_catboost_forecasts_12m.csv

Generating 24-month forecast...
Training model for 24-month horizon...


Generating 24-month predictions...
Forecast saved to canada_catboost_forecasts_24m.csv

Created files:
canada_catboost_forecasts_12m.csv
canada_catboost_forecasts_24m.csv


**Forecasts**
 - **Country: USA**
 - **Forecast Horizons:12M and 24M Forecast**

In [4]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import LightGBMModel, CatBoostModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict

class usaCBForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      """
      Initialize the CatBoost forecast generator for usa data

      Args:
          data_path: Path to the usa data CSV file
          random_seed: Random seed for reproducibility
      """
      # Set random seeds
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      # Load data
      self.data = pd.read_csv(data_path)

      # Define target and exogenous variables
      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      """
      Prepare training data based on forecast horizon

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Tuple of target and exogenous TimeSeries objects
      """
      # Split data based on forecast horizon
      train = self.data.head(-forecast_horizon).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      # Stack target variables
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(train[var]))

      # Stack exogenous variables
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> CatBoostModel:
      """
      Create and return the CatBoost model with horizon-specific parameters

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Configured CatBoostModel
      """
      return CatBoostModel(
          lags= 3, # forecast_horizon,
          lags_past_covariates=3, # forecast_horizon,
          output_chunk_length=forecast_horizon,
          random_state=0
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      """
      Generate forecasts for specified horizons

      Args:
          forecast_horizons: List of forecast horizons (e.g., [12, 24])

      Returns:
          Dictionary with forecast horizons as keys and forecast DataFrames as values
      """
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Create and train model
          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              # verbose=True
          )

          # Generate forecast
          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(n=horizon)

          # Convert to DataFrame
          pred_df = pd.DataFrame({
              'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
              'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
              'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
              'forecast_eer': pred.pd_dataframe().iloc[:, 3],
              'forecast_ir': pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          # Save forecast to CSV
          output_file = f'usa_catboost_forecasts_{horizon}m.csv'
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts

def main():
  """
  Main function to run the LGBM forecast generation
  """
  # Initialize forecast generator
  generator = usaCBForecastGenerator(
      data_path='all_mulvar_data_usa_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"usa_catboost_forecasts_{horizon}m.csv")

if __name__ == "__main__":
  main()

# Created/Modified files during execution:
# usa_catboost_forecasts_12m.csv
# usa_catboost_forecasts_24m.csv


Generating 12-month forecast...
Training model for 12-month horizon...


Generating 12-month predictions...
Forecast saved to usa_catboost_forecasts_12m.csv

Generating 24-month forecast...
Training model for 24-month horizon...


Generating 24-month predictions...
Forecast saved to usa_catboost_forecasts_24m.csv

Created files:
usa_catboost_forecasts_12m.csv
usa_catboost_forecasts_24m.csv


**Forecasts**
 - **Country: France**
 - **Forecast Horizons:12M and 24M Forecast**

In [5]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import LightGBMModel, CatBoostModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict

class franceCBForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      """
      Initialize the CatBoost forecast generator for france data

      Args:
          data_path: Path to the france data CSV file
          random_seed: Random seed for reproducibility
      """
      # Set random seeds
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      # Load data
      self.data = pd.read_csv(data_path)

      # Define target and exogenous variables
      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      """
      Prepare training data based on forecast horizon

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Tuple of target and exogenous TimeSeries objects
      """
      # Split data based on forecast horizon
      train = self.data.head(-forecast_horizon).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      # Stack target variables
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(train[var]))

      # Stack exogenous variables
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> LightGBMModel:
      """
      Create and return the CatBoost model with horizon-specific parameters

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Configured CatBoostModel
      """
      return CatBoostModel(
          lags=3, # forecast_horizon,
          lags_past_covariates=3, # forecast_horizon,
          output_chunk_length=forecast_horizon,
          random_state=0
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      """
      Generate forecasts for specified horizons

      Args:
          forecast_horizons: List of forecast horizons (e.g., [12, 24])

      Returns:
          Dictionary with forecast horizons as keys and forecast DataFrames as values
      """
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Create and train model
          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              # verbose=True
          )

          # Generate forecast
          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(n=horizon)

          # Convert to DataFrame
          pred_df = pd.DataFrame({
              'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
              'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
              'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
              'forecast_eer': pred.pd_dataframe().iloc[:, 3],
              'forecast_ir': pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          # Save forecast to CSV
          output_file = f'france_catboost_forecasts_{horizon}m.csv'
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts

def main():
  """
  Main function to run the CatBoost forecast generation
  """
  # Initialize forecast generator
  generator = franceCBForecastGenerator(
      data_path='all_mulvar_data_france_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"france_catboost_forecasts_{horizon}m.csv")

if __name__ == "__main__":
  main()

# Created/Modified files during execution:
# france_lgbm_forecasts_12m.csv
# france_lgbm_forecasts_24m.csv


Generating 12-month forecast...
Training model for 12-month horizon...


Generating 12-month predictions...
Forecast saved to france_catboost_forecasts_12m.csv

Generating 24-month forecast...
Training model for 24-month horizon...


Generating 24-month predictions...
Forecast saved to france_catboost_forecasts_24m.csv

Created files:
france_catboost_forecasts_12m.csv
france_catboost_forecasts_24m.csv


**Forecasts**
 - **Country: Germany**
 - **Forecast Horizons:12M and 24M Forecast**

In [6]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import LightGBMModel, CatBoostModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict

class germanyCBForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      """
      Initialize the CatBoost forecast generator for germany data

      Args:
          data_path: Path to the germany data CSV file
          random_seed: Random seed for reproducibility
      """
      # Set random seeds
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      # Load data
      self.data = pd.read_csv(data_path)

      # Define target and exogenous variables
      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      """
      Prepare training data based on forecast horizon

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Tuple of target and exogenous TimeSeries objects
      """
      # Split data based on forecast horizon
      train = self.data.head(-forecast_horizon).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      # Stack target variables
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(train[var]))

      # Stack exogenous variables
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> CatBoostModel:
      """
      Create and return the CatBoost model with horizon-specific parameters

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Configured CatBoostModel
      """
      return CatBoostModel(
          lags=3, # forecast_horizon,
          lags_past_covariates=3, # forecast_horizon,
          output_chunk_length=forecast_horizon,
          random_state=0
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      """
      Generate forecasts for specified horizons

      Args:
          forecast_horizons: List of forecast horizons (e.g., [12, 24])

      Returns:
          Dictionary with forecast horizons as keys and forecast DataFrames as values
      """
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Create and train model
          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              # verbose=True
          )

          # Generate forecast
          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(n=horizon)

          # Convert to DataFrame
          pred_df = pd.DataFrame({
              'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
              'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
              'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
              'forecast_eer': pred.pd_dataframe().iloc[:, 3],
              'forecast_ir': pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          # Save forecast to CSV
          output_file = f'germany_catboost_forecasts_{horizon}m.csv'
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts

def main():
  """
  Main function to run the LGBM forecast generation
  """
  # Initialize forecast generator
  generator = germanyCBForecastGenerator(
      data_path='all_mulvar_data_germany_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"germany_catboost_forecasts_{horizon}m.csv")

if __name__ == "__main__":
  main()

# Created/Modified files during execution:
# germany_lgbm_forecasts_12m.csv
# germany_lgbm_forecasts_24m.csv


Generating 12-month forecast...
Training model for 12-month horizon...


Generating 12-month predictions...
Forecast saved to germany_catboost_forecasts_12m.csv

Generating 24-month forecast...
Training model for 24-month horizon...


Generating 24-month predictions...
Forecast saved to germany_catboost_forecasts_24m.csv

Created files:
germany_catboost_forecasts_12m.csv
germany_catboost_forecasts_24m.csv


**Forecasts**
 - **Country: japan**
 - **Forecast Horizons:12M and 24M Forecast**

In [7]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import LightGBMModel, CatBoostModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict

class japanCBForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      """
      Initialize the CatBoost forecast generator for japan data

      Args:
          data_path: Path to the japan data CSV file
          random_seed: Random seed for reproducibility
      """
      # Set random seeds
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      # Load data
      self.data = pd.read_csv(data_path)

      # Define target and exogenous variables
      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      """
      Prepare training data based on forecast horizon

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Tuple of target and exogenous TimeSeries objects
      """
      # Split data based on forecast horizon
      train = self.data.head(-forecast_horizon).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      # Stack target variables
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(train[var]))

      # Stack exogenous variables
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> CatBoostModel:
      """
      Create and return the CatBoost model with horizon-specific parameters

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Configured CatBoostModel
      """
      return CatBoostModel(
          lags=3, # forecast_horizon,
          lags_past_covariates=3, # forecast_horizon,
          output_chunk_length=forecast_horizon,
          random_state=0
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      """
      Generate forecasts for specified horizons

      Args:
          forecast_horizons: List of forecast horizons (e.g., [12, 24])

      Returns:
          Dictionary with forecast horizons as keys and forecast DataFrames as values
      """
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Create and train model
          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              # verbose=True
          )

          # Generate forecast
          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(n=horizon)

          # Convert to DataFrame
          pred_df = pd.DataFrame({
              'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
              'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
              'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
              'forecast_eer': pred.pd_dataframe().iloc[:, 3],
              'forecast_ir': pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          # Save forecast to CSV
          output_file = f'japan_catboost_forecasts_{horizon}m.csv'
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts

def main():
  """
  Main function to run the LGBM forecast generation
  """
  # Initialize forecast generator
  generator = japanCBForecastGenerator(
      data_path='all_mulvar_data_japan_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"japan_catboost_forecasts_{horizon}m.csv")

if __name__ == "__main__":
  main()

# Created/Modified files during execution:
# japan_catboost_forecasts_12m.csv
# japan_catboost_forecasts_24m.csv


Generating 12-month forecast...
Training model for 12-month horizon...


Generating 12-month predictions...
Forecast saved to japan_catboost_forecasts_12m.csv

Generating 24-month forecast...
Training model for 24-month horizon...


Generating 24-month predictions...
Forecast saved to japan_catboost_forecasts_24m.csv

Created files:
japan_catboost_forecasts_12m.csv
japan_catboost_forecasts_24m.csv


**Forecasts**
 - **Country: UK**
 - **Forecast Horizons:12M and 24M Forecast**

In [8]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import LightGBMModel, CatBoostModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict

class ukCBForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      """
      Initialize the CatBoost forecast generator for uk data

      Args:
          data_path: Path to the uk data CSV file
          random_seed: Random seed for reproducibility
      """
      # Set random seeds
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      # Load data
      self.data = pd.read_csv(data_path)

      # Define target and exogenous variables
      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      """
      Prepare training data based on forecast horizon

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Tuple of target and exogenous TimeSeries objects
      """
      # Split data based on forecast horizon
      train = self.data.head(-forecast_horizon).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      # Stack target variables
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(train[var]))

      # Stack exogenous variables
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> CatBoostModel:
      """
      Create and return the CatBoost model with horizon-specific parameters

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Configured CatBoostModel
      """
      return CatBoostModel(
          lags=3, # forecast_horizon,
          lags_past_covariates=3, # forecast_horizon,
          output_chunk_length=forecast_horizon,
          random_state=0
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      """
      Generate forecasts for specified horizons

      Args:
          forecast_horizons: List of forecast horizons (e.g., [12, 24])

      Returns:
          Dictionary with forecast horizons as keys and forecast DataFrames as values
      """
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Create and train model
          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              # verbose=True
          )

          # Generate forecast
          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(n=horizon)

          # Convert to DataFrame
          pred_df = pd.DataFrame({
              'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
              'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
              'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
              'forecast_eer': pred.pd_dataframe().iloc[:, 3],
              'forecast_ir': pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          # Save forecast to CSV
          output_file = f'uk_catboost_forecasts_{horizon}m.csv'
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts

def main():
  """
  Main function to run the LGBM forecast generation
  """
  # Initialize forecast generator
  generator = ukCBForecastGenerator(
      data_path='all_mulvar_data_uk_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"uk_catboost_forecasts_{horizon}m.csv")

if __name__ == "__main__":
  main()

# Created/Modified files during execution:
# uk_lgbm_forecasts_12m.csv
# uk_lgbm_forecasts_24m.csv


Generating 12-month forecast...
Training model for 12-month horizon...


Generating 12-month predictions...
Forecast saved to uk_catboost_forecasts_12m.csv

Generating 24-month forecast...
Training model for 24-month horizon...


Generating 24-month predictions...
Forecast saved to uk_catboost_forecasts_24m.csv

Created files:
uk_catboost_forecasts_12m.csv
uk_catboost_forecasts_24m.csv


**Forecasts**
 - **Country: Italy**
 - **Forecast Horizons:12M and 24M Forecast**

In [9]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import LightGBMModel, CatBoostModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict

class italyCBForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      """
      Initialize the CatBoost forecast generator for italy data

      Args:
          data_path: Path to the italy data CSV file
          random_seed: Random seed for reproducibility
      """
      # Set random seeds
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      # Load data
      self.data = pd.read_csv(data_path)

      # Define target and exogenous variables
      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      """
      Prepare training data based on forecast horizon

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Tuple of target and exogenous TimeSeries objects
      """
      # Split data based on forecast horizon
      train = self.data.head(-forecast_horizon).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      # Stack target variables
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(train[var]))

      # Stack exogenous variables
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> CatBoostModel:
      """
      Create and return the CatBoost model with horizon-specific parameters

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Configured CatBoostModel
      """
      return CatBoostModel(
          lags=3, # forecast_horizon,
          lags_past_covariates=3, # forecast_horizon,
          output_chunk_length=forecast_horizon,
          random_state=0
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      """
      Generate forecasts for specified horizons

      Args:
          forecast_horizons: List of forecast horizons (e.g., [12, 24])

      Returns:
          Dictionary with forecast horizons as keys and forecast DataFrames as values
      """
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Create and train model
          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              # verbose=True
          )

          # Generate forecast
          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(n=horizon)

          # Convert to DataFrame
          pred_df = pd.DataFrame({
              'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
              'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
              'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
              'forecast_eer': pred.pd_dataframe().iloc[:, 3],
              'forecast_ir': pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          # Save forecast to CSV
          output_file = f'italy_catboost_forecasts_{horizon}m.csv'
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts

def main():
  """
  Main function to run the CatBoost forecast generation
  """
  # Initialize forecast generator
  generator = italyCBForecastGenerator(
      data_path='all_mulvar_data_italy_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"italy_catboost_forecasts_{horizon}m.csv")

if __name__ == "__main__":
  main()

# Created/Modified files during execution:
# italy_catboost_forecasts_12m.csv
# italy_catboost_forecasts_24m.csv


Generating 12-month forecast...
Training model for 12-month horizon...


Generating 12-month predictions...
Forecast saved to italy_catboost_forecasts_12m.csv

Generating 24-month forecast...
Training model for 24-month horizon...


Generating 24-month predictions...
Forecast saved to italy_catboost_forecasts_24m.csv

Created files:
italy_catboost_forecasts_12m.csv
italy_catboost_forecasts_24m.csv
